<a href="https://colab.research.google.com/github/dcsuoe-ctrl/multimodal-chinese-speech-analysis/blob/main/align.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [2]:
def create_time_windows(duration, window_size=0.5):

    windows = []

    start = 0.0

    while start < duration:
        end = min(start + window_size, duration)
        windows.append((start, end))
        start += window_size

    return windows

In [4]:
def align_prosody_to_windows(prosody, windows, fps):
    """
    Aggregate frame-level prosodic features
    into temporal windows.
    """

    f0 = np.asarray(prosody["f0"])
    intensity = np.asarray(prosody["intensity"])

    aligned = []

    for start, end in windows:

        start_frame = int(start * fps)
        end_frame = int(end * fps)

        f0_window = f0[start_frame:end_frame]
        intensity_window = intensity[start_frame:end_frame]

        # Remove invalid F0 values
        f0_valid = f0_window[f0_window > 0]

        aligned.append({
            "start": start,
            "end": end,
            "f0_mean": float(np.mean(f0_valid))
                if len(f0_valid) > 0 else np.nan,
            "intensity_mean": float(np.mean(intensity_window))
                if len(intensity_window) > 0 else np.nan
        })

    return aligned

In [5]:
def align_gesture_to_windows(results, windows, fps):
    """
    Aggregate gesture information into temporal windows.
    """

    aligned = []

    for start, end in windows:

        start_frame = int(start * fps)
        end_frame = int(end * fps)

        window_results = results[start_frame:end_frame]

        hand_frames = sum(
            1 for r in window_results
            if r["hands"]
        )

        aligned.append({
            "start": start,
            "end": end,
            "frames_with_hands": hand_frames,
            "hand_presence_ratio": (
                hand_frames / len(window_results)
                if len(window_results) > 0 else 0.0
            )
        })

    return aligned

In [6]:
def align_modalities(prosody, gesture_results, duration, fps,
                     window_size=0.5):
    """
    Align prosody and gesture information
    using a shared temporal grid.
    """

    windows = create_time_windows(
        duration,
        window_size
    )

    prosody_aligned = align_prosody_to_windows(
        prosody,
        windows,
        fps
    )

    gesture_aligned = align_gesture_to_windows(
        gesture_results,
        windows,
        fps
    )

    aligned = []

    for p, g in zip(prosody_aligned, gesture_aligned):

        aligned.append({
            "start": p["start"],
            "end": p["end"],
            "f0_mean": p["f0_mean"],
            "intensity_mean": p["intensity_mean"],
            "frames_with_hands": g["frames_with_hands"],
            "hand_presence_ratio": g["hand_presence_ratio"]
        })

    return aligned